# WeaviateVectorStore Test Notebook

This notebook demonstrates how to use the WeaviateVectorStore class to:
1. Create a tenant
2. Index a document with chunks
3. Query for similar chunks

## Prerequisites

**IMPORTANT**: Before running this notebook:

1. **Weaviate must be running** with the OpenAI API key configured:
   ```bash
   # From project root
   docker-compose down
   docker-compose up -d
   ```

In [1]:
import sys
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

# Add the src directory to Python path
src_path = Path.cwd().parent / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from doc_chat.rag.weaviate_vector_store import (
    WeaviateVectorStore,
    Document,
    DocumentChunk
)

## Clean Up Existing Collections

First, let's delete any existing collections to ensure they're created with the correct multi-tenancy configuration

In [2]:
import weaviate

# Connect to Weaviate and delete existing collections
client = weaviate.use_async_with_local(host="localhost", port=8080)
await client.connect()

# Delete collections if they exist
for collection_name in ["Document", "DocumentChunk"]:
    if await client.collections.exists(collection_name):
        await client.collections.delete(collection_name)
        print(f"✓ Deleted existing '{collection_name}' collection")
    else:
        print(f"  '{collection_name}' collection doesn't exist (skip)")

await client.close()
print("\n✓ Cleanup complete")

✓ Deleted existing 'Document' collection
✓ Deleted existing 'DocumentChunk' collection

✓ Cleanup complete


## Initialize WeaviateVectorStore

Make sure Weaviate is running locally on port 8080 (e.g., via `docker-compose up`)

In [3]:
# Initialize the vector store using async factory method
vector_store = await WeaviateVectorStore.create(embedding_model='text-embedding-3-small')
print("✓ WeaviateVectorStore initialized")
print(f"✓ Client connected: {vector_store.client.is_connected()}")

/Users/patrick/projects/doc-chat/api/venv/lib/python3.13/site-packages/weaviate/warnings.py:196: DeprecationWarning: Dep024: You are using the `vectorizer_config` argument in `collection.config.create()`, which is deprecated.
            Use the `vector_config` argument instead.
            
  warnings.warn(


✓ WeaviateVectorStore initialized
✓ Client connected: True


## Create a Tenant

Multi-tenancy allows us to isolate data per user

In [4]:
# Create a test user tenant
test_user_id = f"test_user_{uuid4().hex[:8]}"
print(f"Creating tenant for user: {test_user_id}")

await vector_store.create_tenant(test_user_id)
print(f"✓ Tenant created for {test_user_id}")

Creating tenant for user: test_user_0a136962
✓ Tenant created for test_user_0a136962


## Create Sample Document and Chunks

We'll create a sample document about Python programming with several chunks

In [5]:
# Create a sample document
doc_id = f"doc_{uuid4().hex[:8]}"
document = Document(
    doc_id=doc_id,
    file_name="python_guide.pdf",
    created_at=datetime.now(timezone.utc),
    num_pages=3
)

# Create sample chunks with different content
chunks = [
    DocumentChunk(
        chunk_id=f"chunk_{uuid4().hex[:8]}",
        doc_id=doc_id,
        page_number=1,
        page_content="Python is a high-level, interpreted programming language known for its simplicity and readability. It supports multiple programming paradigms including procedural, object-oriented, and functional programming.",
        created_at=datetime.now(timezone.utc)
    ),
    DocumentChunk(
        chunk_id=f"chunk_{uuid4().hex[:8]}",
        doc_id=doc_id,
        page_number=1,
        page_content="Python's syntax emphasizes code readability with significant whitespace. The language provides constructs intended to enable writing clear programs on both small and large scales.",
        created_at=datetime.now(timezone.utc)
    ),
    DocumentChunk(
        chunk_id=f"chunk_{uuid4().hex[:8]}",
        doc_id=doc_id,
        page_number=2,
        page_content="Python has a comprehensive standard library that supports many common programming tasks such as connecting to web servers, reading and writing files, and working with data.",
        created_at=datetime.now(timezone.utc)
    ),
    DocumentChunk(
        chunk_id=f"chunk_{uuid4().hex[:8]}",
        doc_id=doc_id,
        page_number=2,
        page_content="Popular Python frameworks include Django and Flask for web development, NumPy and Pandas for data analysis, and TensorFlow and PyTorch for machine learning applications.",
        created_at=datetime.now(timezone.utc)
    ),
    DocumentChunk(
        chunk_id=f"chunk_{uuid4().hex[:8]}",
        doc_id=doc_id,
        page_number=3,
        page_content="Python's dynamic typing and automatic memory management make it easy to learn for beginners while remaining powerful enough for complex applications. It's widely used in web development, data science, automation, and artificial intelligence.",
        created_at=datetime.now(timezone.utc)
    )
]

print(f"✓ Created document '{document.file_name}' with {len(chunks)} chunks")
for i, chunk in enumerate(chunks, 1):
    print(f"  Chunk {i} (page {chunk.page_number}): {chunk.page_content[:60]}...")

✓ Created document 'python_guide.pdf' with 5 chunks
  Chunk 1 (page 1): Python is a high-level, interpreted programming language kno...
  Chunk 2 (page 1): Python's syntax emphasizes code readability with significant...
  Chunk 3 (page 2): Python has a comprehensive standard library that supports ma...
  Chunk 4 (page 2): Popular Python frameworks include Django and Flask for web d...
  Chunk 5 (page 3): Python's dynamic typing and automatic memory management make...


## Index the Document

Now we'll index the document and its chunks in Weaviate

In [6]:
# Index the document and chunks
print(f"Indexing document {doc_id}...")
document_uuid = await vector_store.index_document(
    user_id=test_user_id,
    document=document,
    chunks=chunks
)

print(f"✓ Document indexed successfully!")
print(f"  Document UUID: {document_uuid}")
print(f"  Indexed {len(chunks)} chunks with embeddings")

Indexing document doc_ffd2e713...
✓ Document indexed successfully!
  Document UUID: 1c5cb36d-0a97-4d18-af56-0fb39b990082
  Indexed 5 chunks with embeddings


## Query for Similar Chunks

Let's run some queries to retrieve relevant chunks based on semantic similarity

In [7]:
# Query 1: Search for information about machine learning
query1 = "machine learning frameworks"
print(f"Query: '{query1}'\n")

results1 = await vector_store.search_chunks(
    user_id=test_user_id,
    doc_id=doc_id,
    query=query1,
    k=2
)

print(f"Found {len(results1)} relevant chunks:\n")
for i, chunk in enumerate(results1, 1):
    print(f"Result {i}:")
    print(f"  Page: {chunk.page_number}")
    print(f"  Content: {chunk.page_content}")
    print()

Query: 'machine learning frameworks'

Found 2 relevant chunks:

Result 1:
  Page: 2
  Content: Popular Python frameworks include Django and Flask for web development, NumPy and Pandas for data analysis, and TensorFlow and PyTorch for machine learning applications.

Result 2:
  Page: 3
  Content: Python's dynamic typing and automatic memory management make it easy to learn for beginners while remaining powerful enough for complex applications. It's widely used in web development, data science, automation, and artificial intelligence.



In [8]:
# Query 2: Search for information about Python syntax
query2 = "code readability and syntax"
print(f"Query: '{query2}'\n")

results2 = await vector_store.search_chunks(
    user_id=test_user_id,
    doc_id=doc_id,
    query=query2,
    k=2
)

print(f"Found {len(results2)} relevant chunks:\n")
for i, chunk in enumerate(results2, 1):
    print(f"Result {i}:")
    print(f"  Page: {chunk.page_number}")
    print(f"  Content: {chunk.page_content}")
    print()

Query: 'code readability and syntax'

Found 2 relevant chunks:

Result 1:
  Page: 1
  Content: Python's syntax emphasizes code readability with significant whitespace. The language provides constructs intended to enable writing clear programs on both small and large scales.

Result 2:
  Page: 1
  Content: Python is a high-level, interpreted programming language known for its simplicity and readability. It supports multiple programming paradigms including procedural, object-oriented, and functional programming.



In [9]:
# Query 3: Search for information about web development
query3 = "web servers and files"
print(f"Query: '{query3}'\n")

results3 = await vector_store.search_chunks(
    user_id=test_user_id,
    doc_id=doc_id,
    query=query3,
    k=2
)

print(f"Found {len(results3)} relevant chunks:\n")
for i, chunk in enumerate(results3, 1):
    print(f"Result {i}:")
    print(f"  Page: {chunk.page_number}")
    print(f"  Content: {chunk.page_content}")
    print()

Query: 'web servers and files'

Found 2 relevant chunks:

Result 1:
  Page: 2
  Content: Python has a comprehensive standard library that supports many common programming tasks such as connecting to web servers, reading and writing files, and working with data.

Result 2:
  Page: 2
  Content: Popular Python frameworks include Django and Flask for web development, NumPy and Pandas for data analysis, and TensorFlow and PyTorch for machine learning applications.



## Test Document Retrieval Methods

Let's test the new methods for retrieving documents (not chunks)

In [10]:
# Test get_document - retrieve a specific document by ID
print("Testing get_document method:")
print(f"Looking for document with doc_id: {doc_id}\n")

retrieved_doc = await vector_store.get_document(
    user_id=test_user_id,
    doc_id=doc_id
)

if retrieved_doc:
    print(f"✓ Document found!")
    print(f"  doc_id: {retrieved_doc.doc_id}")
    print(f"  file_name: {retrieved_doc.file_name}")
    print(f"  num_pages: {retrieved_doc.num_pages}")
    print(f"  created_at: {retrieved_doc.created_at}")
else:
    print("✗ Document not found")

# Test with non-existent document
print("\nTesting with non-existent doc_id:")
non_existent_doc = await vector_store.get_document(
    user_id=test_user_id,
    doc_id="non_existent_doc"
)
print(f"Result: {non_existent_doc}")

Testing get_document method:
Looking for document with doc_id: doc_ffd2e713

✓ Document found!
  doc_id: doc_ffd2e713
  file_name: python_guide.pdf
  num_pages: 3
  created_at: 2025-11-29 13:59:15.840453+00:00

Testing with non-existent doc_id:
Result: None


In [11]:
# Create a second document to test get_documents sorting
import asyncio

print("Creating a second document to test get_documents...\n")

# Wait a moment to ensure different timestamps
await asyncio.sleep(1)

doc_id_2 = f"doc_{uuid4().hex[:8]}"
document_2 = Document(
    doc_id=doc_id_2,
    file_name="javascript_guide.pdf",
    created_at=datetime.now(timezone.utc),
    num_pages=2
)

chunks_2 = [
    DocumentChunk(
        chunk_id=f"chunk_{uuid4().hex[:8]}",
        doc_id=doc_id_2,
        page_number=1,
        page_content="JavaScript is a versatile programming language primarily used for web development.",
        created_at=datetime.now(timezone.utc)
    )
]

document_uuid_2 = await vector_store.index_document(
    user_id=test_user_id,
    document=document_2,
    chunks=chunks_2
)

print(f"✓ Second document indexed: {document_2.file_name}")
print(f"  Document UUID: {document_uuid_2}")

Creating a second document to test get_documents...

✓ Second document indexed: javascript_guide.pdf
  Document UUID: 94630043-be83-4c52-81f7-f9808094669a


In [12]:
# Test get_documents - retrieve all documents ordered by date (newest first)
print("Testing get_documents method:\n")

all_docs = await vector_store.get_documents(user_id=test_user_id)

print(f"✓ Found {len(all_docs)} document(s):\n")
for i, doc in enumerate(all_docs, 1):
    print(f"Document {i}:")
    print(f"  doc_id: {doc.doc_id}")
    print(f"  file_name: {doc.file_name}")
    print(f"  num_pages: {doc.num_pages}")
    print(f"  created_at: {doc.created_at}")
    print()

print("✓ Documents are ordered by created_at (newest first)")

Testing get_documents method:

✓ Found 2 document(s):

Document 1:
  doc_id: doc_8adc13c1
  file_name: javascript_guide.pdf
  num_pages: 2
  created_at: 2025-11-29 13:59:21.222881+00:00

Document 2:
  doc_id: doc_ffd2e713
  file_name: python_guide.pdf
  num_pages: 3
  created_at: 2025-11-29 13:59:15.840453+00:00

✓ Documents are ordered by created_at (newest first)


## Cleanup (Optional)

Close the Weaviate client connection when done

In [13]:
# Close the client connection
await vector_store.close()
print("✓ Weaviate client connection closed")

✓ Weaviate client connection closed
